# 05 — SSL Fine-tuning: Taxonomy Classifier\n
\n
Fine-tune the SSL-pretrained backbone for family/genus/species classification on FishNet (top-100 species).\n
\n
This notebook loads the SimCLR checkpoint from notebook 04 and fine-tunes a taxonomy classifier.

In [1]:
from pathlib import Path
import sys
import torch

ROOT = Path('..').resolve()
sys.path.append(str((ROOT / 'src').resolve()))

MANIFEST_DIR = (ROOT / 'data' / 'manifests').resolve()
SSL_DIR = (ROOT / 'outputs' / 'ssl_simclr').resolve()
OUT_DIR = (ROOT / 'outputs' / 'taxonomy_ssl').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)
print('device:', device)
print('OUT_DIR:', OUT_DIR)


device: mps
OUT_DIR: /Users/sushantshah/Desktop/Fish Codes/outputs/taxonomy_ssl


In [ ]:
from fish_ai.data.jsonl import read_jsonl
from fish_ai.data.taxonomy_dataset import FishTaxonomyDataset
from fish_ai.models.taxonomy_classifier import TaxonomyClassifier, TaxonomyHeadSizes
from fish_ai.train.taxonomy_train import TrainConfig, build_label_maps, evaluate, train_one_epoch
from torch.utils.data import DataLoader

train_manifest = MANIFEST_DIR / 'fishnet_taxonomy_train.jsonl'
val_manifest = MANIFEST_DIR / 'fishnet_taxonomy_val.jsonl'

ds_train = FishTaxonomyDataset(train_manifest, image_size=224, augment=True)
ds_val = FishTaxonomyDataset(val_manifest, image_size=224, augment=False)

train_rows = []
for r in read_jsonl(train_manifest):
    tax = r['taxonomy']
    train_rows.append({'family': tax['family'], 'genus': tax['genus'], 'species': tax['species']})

maps = build_label_maps(train_rows)
sizes = TaxonomyHeadSizes(
    n_family=len(maps['family']),
    n_genus=len(maps['genus']),
    n_species=len(maps['species']),
)

sizes


In [ ]:
# Load SSL checkpoint (SimCLR backbone is ResNet-50)
ssl_ckpt = SSL_DIR / 'simclr_resnet50.pt'
print('ssl_ckpt exists:', ssl_ckpt.exists())

# Initialize taxonomy model with the same backbone and load encoder weights
model = TaxonomyClassifier(sizes, backbone='resnet50', pretrained=False).to(device)

ckpt = torch.load(ssl_ckpt, map_location='cpu')
state = ckpt['model_state']

# Map SimCLR encoder weights into taxonomy backbone (both are torchvision resnet50 without fc)
enc_prefix = 'encoder.'
mapped = {k[len(enc_prefix):]: v for k, v in state.items() if k.startswith(enc_prefix)}
missing, unexpected = model.backbone.load_state_dict(mapped, strict=False)
print('loaded backbone; missing:', len(missing), 'unexpected:', len(unexpected))


In [ ]:
# Fine-tune
cfg = TrainConfig(num_epochs=5, batch_size=64, num_workers=2, lr=3e-4)
dl_train = DataLoader(ds_train, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
dl_val = DataLoader(ds_val, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

history = []
for epoch in range(cfg.num_epochs):
    train_metrics = train_one_epoch(model, dl_train, opt, device=device, maps=maps)
    val_metrics = evaluate(model, dl_val, device=device, maps=maps)

    row = {'epoch': epoch + 1, **train_metrics}
    for lvl in ['family', 'genus', 'species']:
        for k, v in val_metrics[lvl].items():
            row[f'val_{lvl}_{k}'] = v

    history.append(row)
    print(row)

ckpt_path = OUT_DIR / 'taxonomy_ssl_resnet50.pt'
torch.save(
    {'model_state': model.state_dict(), 'backbone': model.backbone_name, 'maps': maps, 'cfg': cfg.__dict__},
    ckpt_path,
)
print('saved:', ckpt_path)

from fish_ai.utils.run_logging import write_csv, write_json

write_csv(OUT_DIR / 'history.csv', history)
write_json(OUT_DIR / 'val_metrics_last.json', val_metrics)
